# 00 · Class baseline: RAG_DB for CraveCart

The simple pipeline from class: load the Markdown files → split into chunks → embed → save in Chroma → retrieve → ask Groq. It is kept as the baseline that `01_build_vector_db.ipynb` and `02_chat_app.ipynb` improve on.

Runs in Colab (add `GROQ_API_KEY` under 🔑 Secrets) or locally (key in `.env`).

Install libraries

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -U -q langchain-chroma langchain-huggingface langchain-community langchain-text-splitters langchain-groq
# Locally: pip install langchain-chroma langchain-huggingface langchain-community langchain-groq python-dotenv

Imports

In [ ]:
import os
import glob
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

Get the knowledge base

In Colab this clones the project from GitHub. Set `USE_DRIVE = True` to use your own copy in `MyDrive/ai/knowledge-base` instead, as in class.

In [ ]:
USE_DRIVE = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    project = "/content/drive/MyDrive/ai"
elif IN_COLAB:
    if not os.path.exists("ai-engineering-projects"):
        !git clone --depth 1 https://github.com/C0derANANT/ai-engineering-projects.git
    project = "ai-engineering-projects/03-Food-Delivery-RAG"
else:
    project = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

kb_dir = f"{project}/knowledge-base"
db_name = f"{project}/rag_vector_db" if USE_DRIVE else f"{project}/class_baseline_vector_db"
print("Knowledge base:", kb_dir)

Load knowledge base

In [ ]:
# Load the knowledge base: each subfolder name becomes the doc_type
folders = [f for f in glob.glob(f'{kb_dir}/*') if os.path.isdir(f)]  # skip loose files such as a README

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(
        folder,
        glob='**/*.md',
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'}
    )
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc)

print(f'Loaded {len(documents)} documents')

Split documents into chunks

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print(f'Divided into {len(chunks)} chunks')

Create embeddings and Chroma database

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name='all-MiniLM-L6-v2'
)

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(f'Vectorstore created with {vectorstore._collection.count()} documents')

Testing

In [ ]:
# Testing: which chunks come back for a question?
retriever = vectorstore.as_retriever()
question = "How long does a refund take if I paid with UPI?"
result = retriever.invoke(question)
for doc in result:
    print(doc.metadata['doc_type'], '|', os.path.relpath(doc.metadata['source'], kb_dir))

context = "\n\n".join(content.page_content for content in result)

In [ ]:
SYSTEM_PROMPT = f"""
You are a knowledgeable, friendly assistant representing CraveCart, a food delivery company.
You are chatting with a customer or employee about CraveCart.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

Ask Groq

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

if IN_COLAB:
    from google.colab import userdata
    key = userdata.get('GROQ_API_KEY')
else:
    from dotenv import load_dotenv
    load_dotenv(f'{project}/.env')
    key = os.environ.get('GROQ_API_KEY')

llm = ChatGroq(temperature=0, model_name='openai/gpt-oss-120b', api_key=key)

response = llm.invoke([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)])
print(response.content)

Next: `01_build_vector_db.ipynb` builds a better database (heading-aware chunks, BGE embeddings), and `02_chat_app.ipynb` runs the full Crave Assist app with citations, follow-ups and exact refund maths.